# APUVA SDV — ETL sampai Evaluasi Model (Google Colab)Notebook ini menjalankan alur yang sama dengan dashboard Streamlit:**source-data.xlsx → ETL → fitur → walk-forward evaluation → model terbaik per leaf → ekspor**## Prinsip: memakai kode repo, bukan menyalinnyaSemua pemodelan di sini memanggil **modul repo yang sama** dengan yang dipakaihalaman Streamlit (`etl.pipeline`, `utils.forecasting`, `utils.feature_engineering_optimized`).Jadi angka yang keluar dari notebook ini sama dengan angka di dashboard.Yang **tidak** diambil dari repo hanyalah *loop* walk-forward-nya, karena diStreamlit loop itu menyatu dengan UI (progress bar, checkpoint, session state).Loop di notebook ini ditulis mengikuti `pages/5_Evaluasi.py` dengan parameteryang sama persis. Kalau halaman itu diubah di kemudian hari, notebook ini bisatertinggal — periksa bagian **Konfigurasi** kalau hasilnya terasa berbeda.## Yang Anda perlukan1. **Kode repo** — ZIP repo, atau akses `git clone`.2. **`source-data.xlsx`** — 4 sheet: `Korporasi`, `PTMN`, `Asing`, `Individu`.> ⏱️ Evaluasi recursive memakan waktu. Perkiraan ada di bagian Konfigurasi.> Aktifkan **Runtime → Change runtime type → CPU** (GPU tidak membantu di sini;> model dominan berbasis pohon dan ARIMA).

## 1. Pasang dependensi

In [ ]:
%%capture!pip install -q "pandas>=2.0" "numpy>=1.24,<2.0" "scikit-learn>=1.3" "scipy>=1.10" \               "xgboost>=1.7" "lightgbm>=4.0" "prophet>=1.1" "statsmodels>=0.14" \               "openpyxl>=3.1" plotly# LSTM bersifat opsional. Lewati baris ini kalau tidak ingin memakainya.!pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import sys, os, importlibprint("Python", sys.version.split()[0])for m in ["pandas", "numpy", "sklearn", "xgboost", "lightgbm", "prophet", "statsmodels", "scipy"]:    try:        print(f"  {m:12s} {importlib.import_module(m).__version__}")    except Exception as e:        print(f"  {m:12s} GAGAL: {e}")try:    import torch; print(f"  {'torch':12s} {torch.__version__}  (LSTM tersedia)")except ImportError:    print(f"  {'torch':12s} tidak ada  (LSTM akan dilewati)")

## 2. Ambil kode repoPilih **salah satu** cara di bawah. Cara A paling sederhana dan tidak butuh kredensial.

In [ ]:
# ============ CARA A: unggah ZIP repo (disarankan) ============# Di komputer Anda: kompres folder repo jadi .zip, lalu jalankan sel ini.from google.colab import filesimport zipfile, pathlib, shutil, osup = files.upload()                      # pilih file .zip repozname = list(up.keys())[0]with zipfile.ZipFile(zname) as z:    z.extractall('/content/_unzip')# Cari folder yang memuat etl/pipeline.py (ZIP sering punya folder pembungkus)root = Nonefor p in pathlib.Path('/content/_unzip').rglob('etl/pipeline.py'):    root = p.parent.parent    breakassert root is not None, "Tidak menemukan etl/pipeline.py di dalam ZIP. Pastikan yang dikompres adalah folder repo."REPO = '/content/jbv'if os.path.exists(REPO):    shutil.rmtree(REPO)shutil.move(str(root), REPO)print("Repo siap di", REPO)

In [ ]:
# ============ CARA B: git clone (repo privat butuh token) ============# Buat Personal Access Token di GitHub -> Settings -> Developer settings.# JANGAN menuliskan token langsung di sel; pakai getpass agar tidak tersimpan# di output notebook.## from getpass import getpass# import subprocess# TOKEN = getpass("GitHub token: ")# !git clone -q https://{TOKEN}@github.com/cielchan88/jbv.git /content/jbv# REPO = '/content/jbv'# print("Repo siap di", REPO)

In [ ]:
import sys, osos.chdir(REPO)if REPO not in sys.path:    sys.path.insert(0, REPO)os.makedirs('data/raw', exist_ok=True)os.makedirs('data/processed', exist_ok=True)print("Direktori kerja:", os.getcwd())print("Isi:", sorted(os.listdir('.'))[:12])

## 3. Unggah `source-data.xlsx`

In [ ]:
from google.colab import filesimport pandas as pd, shutilup = files.upload()                      # pilih source-data.xlsxsrc = list(up.keys())[0]shutil.move(src, 'data/raw/source-data.xlsx')xl = pd.ExcelFile('data/raw/source-data.xlsx')print("Sheet ditemukan:", xl.sheet_names)wajib = {'Korporasi', 'PTMN', 'Asing', 'Individu'}kurang = wajib - set(xl.sheet_names)print("❌ Sheet kurang:", kurang) if kurang else print("✅ Keempat sheet lengkap")

## 4. Jalankan ETL

In [ ]:
import logging, timelogging.basicConfig(level=logging.INFO, format='%(message)s', force=True)from etl.pipeline import run_pipelinet0 = time.time()run_pipeline()print(f"\nETL selesai dalam {time.time()-t0:.0f} detik")

In [ ]:
import pandas as pd, numpy as npdf = pd.read_csv('data/processed/sdv-wide.csv')META = ['Row_ID', 'Row_Label', 'Level']time_cols = [c for c in df.columns if c not in META]dates_all = pd.to_datetime(time_cols)def children_of(pid):    if pid == 'D':        return ['A', 'B', 'C']    return [r for r in df.Row_ID            if r.startswith(pid + '.') and r != pid and r.count('.') == pid.count('.') + 1]leaf_nodes = [r for r in df.Row_ID if len(children_of(r)) == 0]print(f"{len(df)} baris, {len(time_cols)} tanggal")print(f"Periode : {time_cols[0]} s/d {time_cols[-1]}")print(f"Leaf    : {len(leaf_nodes)} -> {leaf_nodes}")# Identitas akuntansi harus eksak; kalau tidak, ETL bermasalahD = np.nan_to_num(df[df.Row_ID == 'D'][time_cols].to_numpy(dtype=float).ravel())ABC = sum(np.nan_to_num(df[df.Row_ID == k][time_cols].to_numpy(dtype=float).ravel())          for k in ['A', 'B', 'C'])print(f"|D - (A+B+C)| maks = {np.abs(D - ABC).max():.6f}")

## 5. Konfigurasi evaluasiNilai default di bawah **sama persis** dengan default halaman Evaluasi.| Parameter | Default | Arti ||---|---|---|| `TEST_SIZE` | 20 | persen data untuk periode uji || `HORIZON` | 60 | hari kerja yang dinilai per jendela || `N_WINDOWS` | 3 | jumlah jendela walk-forward || `RECURSIVE` | `True` | prediksi seperti produksi |**Tentang `RECURSIVE`.** Biarkan `True`. Mode `False` memberi model nilai lagdari data aktual periode uji — jauh lebih cepat, tapi metriknya optimistis danperingkat modelnya bisa berbeda. Pengukuran pada data ini: mode directmembuat MAE tampak 13–39% lebih baik daripada yang bisa dicapai produksi, danmengubah model terbaik di 8 dari 18 leaf.

In [ ]:
TEST_SIZE  = 20HORIZON    = 60N_WINDOWS  = 3MIN_TRAIN  = 300RECURSIVE  = True# Pilih model. Buang dari daftar untuk mempercepat.MODELS = ['Naive', 'NaiveMean', 'AutoARIMA', 'VAR', 'Prophet',          'APUVA', 'RandomForest', 'LightGBM', 'XGBoost']try:    import torch  # noqa    MODELS.append('LSTM')except ImportError:    print("torch tidak ada - LSTM dilewati")# Fitur cross-series default MATI untuk model pohon - bukan karena tidak# terpilih, melainkan karena TIDAK BISA DIPAKAI.## Fitur ext_* memang terpilih (sampai 12 dari 25 slot pada A.2.c), tapi# forecaster membangunnya saat fit() dan TIDAK meneruskan external_series saat# predict(), sehingga fitur itu diisi NOL ketika meramal. Model dilatih# mengharapkan nilai nyata lalu diberi nol. Terukur merugikan: LightGBM naik# dari MAE 38,73 menjadi 45,01 saat cross-series diteruskan.## Akarnya bukan sekadar argumen yang lupa diteruskan: pada peramalan# multi-langkah, nilai deret lain di tanggal masa depan juga tidak diketahui -# memakainya sebagai prediktor berarti harus meramalkannya juga.## PENGECUALIAN: VAR tetap mendapatkannya otomatis, karena memang membutuhkan# deret lain secara konstruksi dan menanganinya lewat sistem persamaannya# sendiri. Tanpa itu VAR jatuh jadi random walk (MAE identik Naive di 54/54).USE_CROSS_SERIES = FalseLEAVES = leaf_nodes          # atau mis. leaf_nodes[:3] untuk uji cepat_n_unit = len(LEAVES) * N_WINDOWS_per_unit = (14 if RECURSIVE else 1) * len([m for m in MODELS if m in             ('RandomForest', 'LightGBM', 'XGBoost')]) \            + 11 * ('LSTM' in MODELS) + 49 * ('AutoARIMA' in MODELS)print(f"{_n_unit} unit (leaf x jendela) x {len(MODELS)} model")print(f"Perkiraan kasar: ~{_n_unit * _per_unit / 60:.0f} menit")

## 6. Jalankan walk-forward evaluation

In [ ]:
import warnings, timewarnings.filterwarnings('ignore')import numpy as np, pandas as pdfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scorefrom utils import load_holidays, ML_START_DATEfrom utils.forecasting import forecast_single_seriesfrom utils.feature_engineering_optimized import (    create_features_optimized, select_top_features_optimized,    calculate_series_correlations, select_top_correlated_series,    prepare_external_series_data)from utils.external_loader import load_and_merge_external_featuresholidays_list = load_holidays()time_cols_ml = (time_cols if ML_START_DATE is None                else [c for c in time_cols if pd.to_datetime(c) >= pd.Timestamp(ML_START_DATE)])dates_ml = pd.to_datetime(time_cols_ml)print(f"Periode ML: {time_cols_ml[0]} s/d {time_cols_ml[-1]} ({len(time_cols_ml)} hari)")# VAR WAJIB dapat cross-series. Tanpa deret lain, Vector AutoRegression tidak# punya apa pun untuk diregresikan dan jatuh jadi random walk - terbukti pada# data ini: MAE-nya identik dengan Naive di 54 dari 54 unit, sampai desimal# terakhir. Jadi peta cross-series tetap dihitung kalau VAR ikut dipilih,# berapa pun nilai USE_CROSS_SERIES._perlu_cross = USE_CROSS_SERIES or ('VAR' in MODELS)cross_map = {}if _perlu_cross:    if not USE_CROSS_SERIES:        print("Cross-series dihitung karena VAR dipilih (VAR tidak berfungsi tanpanya).")    t0 = time.time()    for lid in LEAVES:        corr = calculate_series_correlations(df, lid, [l for l in leaf_nodes if l != lid], time_cols_ml)        cs = prepare_external_series_data(df, select_top_correlated_series(corr, top_k=30), time_cols_ml)        cross_map[lid] = load_and_merge_external_features(cs, time_cols_ml)    print(f"Cross-series siap ({time.time()-t0:.0f}s)")def calculate_metrics(actual, predictions, train_values=None):    # Sama persis dengan pages/5_Evaluasi.py.    # Mengembalikan None kalau prediksi tak layak dinilai - sengaja tidak diisi 0,    # karena model gagal yang diisi 0 bisa ikut terpilih sebagai 'model terbaik'.    n = min(len(actual), len(predictions))    if n == 0:        return None    a = np.asarray(actual[:n], dtype=float)    p = np.asarray(predictions[:n], dtype=float)    if not np.all(np.isfinite(p)) or not np.all(np.isfinite(a)):        return None    mae = mean_absolute_error(a, p)    nz = a != 0    mase = np.nan    if train_values is not None:        tv = np.asarray(train_values, dtype=float)        tv = tv[np.isfinite(tv)]        if len(tv) > 1:            sc = np.mean(np.abs(np.diff(tv)))            if sc > 1e-9:                mase = mae / sc    return {        'MAE': mae,        'RMSE': float(np.sqrt(mean_squared_error(a, p))),        'MAPE': float(np.mean(np.abs((a[nz] - p[nz]) / a[nz])) * 100) if nz.sum() else np.nan,        'SMAPE': float(np.mean(np.abs(p - a) / ((np.abs(a) + np.abs(p)) / 2 + 1e-8)) * 100),        'R2': r2_score(a, p),        'DA': float(np.mean(np.sign(np.diff(a)) == np.sign(np.diff(p))) * 100) if n > 1 else np.nan,        'MASE': mase,        'Bias': float(np.mean(p - a)),    }

In [ ]:
rows, failed, preds_store = [], [], {}t0 = time.time()total = len(LEAVES) * N_WINDOWSfor i_leaf, leaf in enumerate(LEAVES):    row = df[df.Row_ID == leaf].iloc[0]    v_full = np.nan_to_num(df[df.Row_ID == leaf][time_cols].to_numpy(dtype=float).ravel())    v_ml = v_full[-len(time_cols_ml):]    anchor_ml = int(len(v_ml) * (1 - TEST_SIZE / 100))    anchor_ap = int(len(v_full) * (1 - TEST_SIZE / 100))    for w in range(N_WINDOWS):        s_ml, e_ml = anchor_ml - w * HORIZON - HORIZON, anchor_ml - w * HORIZON        s_ap, e_ap = anchor_ap - w * HORIZON - HORIZON, anchor_ap - w * HORIZON        if s_ml < MIN_TRAIN or s_ap < MIN_TRAIN:            continue        tr_ml, te_ml = v_ml[:s_ml], v_ml[s_ml:e_ml]        d_ml = [x.strftime('%Y-%m-%d') for x in dates_ml[:s_ml]]        tr_ap, te_ap = v_full[:s_ap], v_full[s_ap:e_ap]        d_ap = [x.strftime('%Y-%m-%d') for x in dates_all[:s_ap]]        for m in MODELS:            # APUVA memakai histori penuh; model lain memakai periode ML            use_full = (m == 'APUVA')            # VAR selalu diberi cross-series (lihat catatan di sel sebelumnya);            # model lain mengikuti USE_CROSS_SERIES.            ext = cross_map.get(leaf) if (USE_CROSS_SERIES or m == 'VAR') else None            dd, tt, aa = (d_ap, tr_ap, te_ap) if use_full else (d_ml, tr_ml, te_ml)            try:                f, _ = forecast_single_series(dates=dd, values=tt, model_name=m,                                              n_days=len(aa), holidays=holidays_list,                                              external_series=ext, row_id=leaf)                f = np.asarray(f, dtype=float)                if len(f) > len(aa):                    f = f[:len(aa)]                elif len(f) < len(aa):                    f = np.pad(f, (0, len(aa) - len(f)), mode='edge')                mt = calculate_metrics(aa, f, train_values=tt)                if mt is None:                    failed.append({'Row_ID': leaf, 'Jendela': w, 'Model': m,                                   'Alasan': 'Prediksi mengandung NaN/inf'})                    continue                mt.update(Row_ID=leaf, Row_Label=row['Row_Label'], Window=w, Model=m)                rows.append(mt)                preds_store[(leaf, w, m)] = (                    (dates_all[s_ap:e_ap] if use_full else dates_ml[s_ml:e_ml]), aa, f)            except Exception as e:                failed.append({'Row_ID': leaf, 'Jendela': w, 'Model': m,                               'Alasan': str(e)[:150]})    done = (i_leaf + 1) * N_WINDOWS    el = time.time() - t0    print(f"[{done:3d}/{total}] {leaf:8s}  {el/60:5.1f} mnt terpakai, "          f"~{el/done*(total-done)/60:5.1f} mnt lagi", flush=True)res = pd.DataFrame(rows)print(f"\nSelesai: {len(res)} hasil, {len(failed)} gagal, {(time.time()-t0)/60:.1f} menit")if failed:    display(pd.DataFrame(failed).head(20))

## 7. Agregasi dan model terbaik per leafMetrik diagregasi dengan **median** lintas jendela, bukan rata-rata: satu jendelaburuk (misalnya periode dengan lonjakan ekstrem) tidak boleh menentukan pemenangsendirian.

In [ ]:
METRICS = ['MAE', 'RMSE', 'MAPE', 'SMAPE', 'R2', 'DA', 'MASE', 'Bias']agg = (res.groupby(['Row_ID', 'Row_Label', 'Model'], as_index=False)[METRICS].median())print("=== Performa keseluruhan (rata-rata lintas leaf) ===")overall = agg.groupby('Model')[METRICS].mean().sort_values('MAE')display(overall.round(3))SELECTION_METRIC = 'MAE'best = agg.loc[agg.groupby('Row_ID')[SELECTION_METRIC].idxmin()].sort_values('Row_ID')print(f"\n=== Model terbaik per leaf (menurut {SELECTION_METRIC}) ===")display(best[['Row_ID', 'Row_Label', 'Model', 'MAE', 'R2', 'DA', 'MASE']].round(3))print(best['Model'].value_counts().to_string())# Garis acuan: model yang tidak mengalahkan ini tidak layak dipakaifor b in ['Naive', 'NaiveMean']:    if b in overall.index:        n_win = int((overall['MAE'] < overall.loc[b, 'MAE']).sum())        print(f"\nModel yang mengalahkan {b} (MAE {overall.loc[b,'MAE']:.2f}): {n_win}")

## 8. Grafik prediksi per leaf

In [ ]:
import plotly.graph_objects as goPLOT_LEAF = LEAVES[0]     # ganti sesuai kebutuhanPLOT_WIN = 0CTX = 60                  # hari histori sebelum periode ujifig = go.Figure()kunci = [(l, w, m) for (l, w, m) in preds_store if l == PLOT_LEAF and w == PLOT_WIN]if kunci:    d0, a0, _ = preds_store[kunci[0]]    hist = [c for c in time_cols if pd.to_datetime(c) < d0[0]][-CTX:]    if hist:        fig.add_trace(go.Scatter(            x=pd.to_datetime(hist),            y=pd.to_numeric(df[df.Row_ID == PLOT_LEAF][hist].to_numpy().ravel()),            mode='lines', name='Histori', line=dict(color='#9aa5ad', width=1.5)))    fig.add_trace(go.Scatter(x=d0, y=a0, mode='lines+markers', name='AKTUAL',                             line=dict(color='#111418', width=3), marker=dict(size=4)))    for (l, w, m) in kunci:        dd, _, ff = preds_store[(l, w, m)]        mae = res[(res.Row_ID == l) & (res.Window == w) & (res.Model == m)]['MAE']        lab = f"{m} (MAE {mae.iloc[0]:.1f})" if len(mae) else m        fig.add_trace(go.Scatter(x=dd, y=ff, mode='lines', name=lab, line=dict(width=1.8)))    fig.update_layout(title=f"{PLOT_LEAF} - jendela {PLOT_WIN}", height=520,                      xaxis_title="Tanggal", yaxis_title="Nilai (USD Juta)",                      hovermode='x unified',                      legend=dict(orientation='h', y=-0.2, yanchor='top'))    fig.show()    print("Garis hitam = aktual. Yang diplot prediksi OUT-OF-SAMPLE, bukan fitted value in-sample.")else:    print("Tidak ada prediksi tersimpan untuk kombinasi ini.")

## 9. Ekspor hasil

In [ ]:
from google.colab import filesfrom datetime import datetimeout = f"evaluasi_{datetime.now().strftime('%Y%m%dT%H%M')}.xlsx"with pd.ExcelWriter(out, engine='openpyxl') as xw:    res.to_excel(xw, index=False, sheet_name='Hasil per Jendela')    agg.to_excel(xw, index=False, sheet_name='Agregat per Leaf')    overall.reset_index().to_excel(xw, index=False, sheet_name='Ringkasan Model')    best.to_excel(xw, index=False, sheet_name='Model Terbaik')    if failed:        pd.DataFrame(failed).to_excel(xw, index=False, sheet_name='Gagal')    pd.DataFrame([{        'test_size': TEST_SIZE, 'horizon': HORIZON, 'n_windows': N_WINDOWS,        'recursive': RECURSIVE, 'cross_series': USE_CROSS_SERIES,        'models': ', '.join(MODELS), 'n_leaf': len(LEAVES),        'periode_ml': f"{time_cols_ml[0]} s/d {time_cols_ml[-1]}",        'dijalankan': datetime.now().isoformat(timespec='seconds'),    }]).T.reset_index().to_excel(xw, index=False, sheet_name='Konfigurasi')print("Tersimpan:", out)files.download(out)

---## Catatan**Sheet Konfigurasi ikut diekspor** agar setiap file hasil bisa ditelusuriparameternya. Hasil dengan `recursive` berbeda tidak boleh dibandingkan langsung.**Kalau ingin lebih cepat**, atur di bagian Konfigurasi: kurangi `MODELS`(buang `AutoARIMA` yang paling lambat, ~49 detik per unit), kurangi `N_WINDOWS`,atau batasi `LEAVES` ke beberapa leaf saja untuk uji coba.**Colab bisa memutus runtime** pada sesi panjang. Untuk 18 leaf × 3 jendela ×seluruh model, pertimbangkan menjalankan bertahap dengan membagi `LEAVES` danmenggabungkan hasilnya, atau memakai Colab Pro.**Notebook ini tidak menyalin logika pemodelan** — semuanya memanggil modul repo.Satu-satunya bagian yang ditulis ulang adalah loop walk-forward dan`calculate_metrics`, yang mengikuti `pages/5_Evaluasi.py` dengan parameter yangsama. Kalau halaman itu berubah, periksa kembali bagian ini.